In [1]:
import sys, os
from pathlib import Path

ROOT = Path(r"C:\Users\user\doc-asistant\doc-asistant")
sys.path.append(str(ROOT))
sys.path.append(str(ROOT / "notebooks" / "from_src"))

import torch
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("faiss:", faiss.__version__)
print("numpy:", np.__version__)
print("sentence-transformers:", SentenceTransformer.__module__)
print("ROOT:", ROOT)
print("cwd:", os.getcwd())

torch: 2.6.0+cu124
cuda: True
faiss: 1.15.1
numpy: 2.5.3
sentence-transformers: sentence_transformers.sentence_transformer.model
ROOT: C:\Users\user\doc-asistant\doc-asistant
cwd: C:\Users\user\doc-asistant\doc-asistant\notebooks


In [2]:
# режем текст на чанки
def chunk_text(text, size=500, overlap=50):
    text = str(text).strip()
    if not text:
        return []
    chunks = []
    start = 0
    step = size - overlap
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start += step
    return chunks

# тест: 1200 символов
sample = "a" * 1200
chunks = chunk_text(sample, size=500, overlap=50)
print("test: 1200 chars ->", len(chunks), "chunks")
print("sizes:", [len(c) for c in chunks])

test: 1200 chars -> 3 chunks
sizes: [500, 500, 300]


In [3]:
# загрузка рецептов
CSV = ROOT / "data" / "cooking_clean.csv"
df = pd.read_csv(CSV, sep=";")
print("recipes:", len(df))
print("columns:", df.columns.tolist())

# склеиваем title + instructions
df["full_text"] = (
    "Recipe: " + df["title"].astype(str) + "\n" +
    "Instructions: " + df["instructions"].astype(str)
)

# чанкинг всех рецептов
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

chunks = []
chunk_meta = []

for idx, row in df.iterrows():
    recipe_chunks = chunk_text(row["full_text"], size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
    for j, c in enumerate(recipe_chunks):
        chunks.append(c)
        chunk_meta.append({
            "recipe_number": int(row["number"]),
            "recipe_title": str(row["title"]),
            "chunk_index": j,
        })

print("total chunks:", len(chunks))
print("avg chunks/recipe:", round(len(chunks) / len(df), 2))
print()
print("first 3 chunks:")
for i in [0, 1, 2]:
    print(f"chunk {i} meta:", chunk_meta[i])
    print(f"chunk {i} text:", chunks[i][:200])
    print()

recipes: 500
columns: ['number', 'title', 'instructions']
total chunks: 1053
avg chunks/recipe: 2.11

first 3 chunks:
chunk 0 meta: {'recipe_number': 1, 'recipe_title': 'cheeseburger potato soup', 'chunk_index': 0}
chunk 0 text: Recipe: cheeseburger potato soup
Instructions: 6 baking potatoes 1 lb of extra lean ground beef 2/3 c butter or margarine 6 c milk 3/4 tsp salt 1/2 tsp pepper 1 1/2 c 6 oz shredded cheddar cheese divi

chunk 1 meta: {'recipe_number': 1, 'recipe_title': 'cheeseburger potato soup', 'chunk_index': 1}
chunk 1 text: otatoes should be soft ready to eat let them cool enough to handle cut in half lengthwise scoop out pulp and reserve discard shells brown ground beef until done drain any grease from the meat set asid

chunk 2 meta: {'recipe_number': 1, 'recipe_title': 'cheeseburger potato soup', 'chunk_index': 2}
chunk 2 text: ground beef salt pepper 1 cup of cheese 2 tablespoons of green onion and 1/2 cup of bacon cook until heated do not boil stir in sour cream if des

In [4]:
# мультиязычный энкодер (работает и на русском, и на английском)
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"

encoder = SentenceTransformer(MODEL_NAME, device=device)
print("model:", MODEL_NAME)
print("device:", encoder.device)
print("max_seq_length:", encoder.max_seq_length)
print("embedding dim:", encoder.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

C:\Users\user\doc-asistant\doc-asistant\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
device: cuda:0
max_seq_length: 128
embedding dim: 384


C:\Users\user\AppData\Local\Temp\ipykernel_5340\3701755321.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dim:", encoder.get_sentence_embedding_dimension())


In [5]:
# эмбеддинги всех 1053 чанков
print("encoding", len(chunks), "chunks...")
embeddings = encoder.encode(
    chunks,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # для cosine через inner product
)

print("embeddings shape:", embeddings.shape)
print("dtype:", embeddings.dtype)
print("norms (first 3):", np.linalg.norm(embeddings[:3], axis=1))

encoding 1053 chunks...


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

embeddings shape: (1053, 384)
dtype: float32
norms (first 3): [1. 1. 1.]


In [6]:
# создаём FAISS-индекс: inner product = cosine для нормализованных векторов
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # IP = inner product
index.add(embeddings.astype("float32"))

print("index total vectors:", index.ntotal)
print("index dim:", index.d)

index total vectors: 1053
index dim: 384


In [7]:
import json
from pathlib import Path

# пути для сохранения
INDEX_PATH = ROOT / "models" / "faiss_recipes.index"
META_PATH = ROOT / "models" / "faiss_recipes_meta.json"

# сохранить FAISS-индекс
faiss.write_index(index, str(INDEX_PATH))
print("saved index:", INDEX_PATH, INDEX_PATH.stat().st_size, "bytes")

# сохранить метаданные чанков (заголовки, номера рецептов)
with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "encoder": MODEL_NAME,
        "num_chunks": len(chunks),
        "meta": chunk_meta,
    }, f, ensure_ascii=False, indent=2)
print("saved meta:", META_PATH, META_PATH.stat().st_size, "bytes")

saved index: C:\Users\user\doc-asistant\doc-asistant\models\faiss_recipes.index 1617453 bytes
saved meta: C:\Users\user\doc-asistant\doc-asistant\models\faiss_recipes_meta.json 122032 bytes


In [9]:
# функция поиска top-k
def search(query, k=3):
    q_emb = encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, indices = index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "idx": int(idx),
            "meta": chunk_meta[idx],
            "text": chunks[idx],
        })
    return results

# тестовые запросы
test_queries = [
    "chocolate cake",
    "шоколадный торт",
    "apple pie with cinnamon",
    "рецепт с яблоками",
]

for q in test_queries:
    print(f"\nQuery: {q}")
    for r in search(q, k=3):
        print(f"  score={r['score']:.3f}  recipe='{r['meta']['recipe_title']}'  chunk={r['meta']['chunk_index']}")
        print(f"  text: {r['text'][:120]}...")


Query: chocolate cake
  score=0.712  recipe='ultimate chocolate cake'  chunk=0
  text: Recipe: ultimate chocolate cake
Instructions: 3 oz unsweetened chocolate 1 c unsalted butter 2 1/3 c firmly packed light...
  score=0.656  recipe='betsys chocolate fudge cake'  chunk=0
  text: Recipe: betsys chocolate fudge cake
Instructions: 3 sq unsweetened chocolate 2 1/4 c sifted flour 2 tsp baking powder 1/...
  score=0.645  recipe='chocolate eclair cake'  chunk=0
  text: Recipe: chocolate eclair cake
Instructions: 1 box graham crackers 2 boxes instant vanilla pudding 1 c powdered sugar 3 c...

Query: шоколадный торт
  score=0.724  recipe='ultimate chocolate cake'  chunk=0
  text: Recipe: ultimate chocolate cake
Instructions: 3 oz unsweetened chocolate 1 c unsalted butter 2 1/3 c firmly packed light...
  score=0.666  recipe='betsys chocolate fudge cake'  chunk=0
  text: Recipe: betsys chocolate fudge cake
Instructions: 3 sq unsweetened chocolate 2 1/4 c sifted flour 2 tsp baking powder 1/...
  

In [10]:
# RAG: берём top-k чанков и склеиваем в контекст
def build_context(query, k=3):
    results = search(query, k=k)
    context_parts = []
    for i, r in enumerate(results, 1):
        context_parts.append(
            f"[{i}] Recipe: {r['meta']['recipe_title']}\n{r['text']}"
        )
    return "\n\n".join(context_parts), results


# проверка: что попадает в контекст для запроса
query = "шоколадный торт"
context, results = build_context(query, k=3)
print(f"Query: {query}")
print(f"Context length: {len(context)} chars")
print(f"Sources: {[r['meta']['recipe_title'] for r in results]}")
print()
print("Context preview:")
print(context[:600])

Query: шоколадный торт
Context length: 1614 chars
Sources: ['ultimate chocolate cake', 'betsys chocolate fudge cake', 'chocolate eclair cake']

Context preview:
[1] Recipe: ultimate chocolate cake
Recipe: ultimate chocolate cake
Instructions: 3 oz unsweetened chocolate 1 c unsalted butter 2 1/3 c firmly packed light brown sugar 2 1/3 c sifted cake flour 2/3 c buttermilk 1 c boiling water 1 tsp vanilla 3 large eggs 2 tsp baking soda 1 tsp salt heat oven to 350u00b0 brush cake pans with melted butter and dust with sifted flour shaking out excess chill until needed melt chocolate in the top of a double boiler allow to cool cream together butter and sugar add eggs one at a time beating well m

[2] Recipe: betsys chocolate fudge cake
Recipe: betsys chocola


In [11]:
# подключаем клиент к Ollama из from_src
sys.path.append(str(ROOT / "notebooks" / "from_src"))
import llm_client
from openai import OpenAI

client = OpenAI(
    base_url=llm_client.OLLAMA_BASE_URL,
    api_key=llm_client.OLLAMA_API_KEY,
)
print("Ollama client ready")
print("model:", llm_client.OLLAMA_MODEL)

Ollama client ready
model: gpt-oss:120b


In [12]:
# RAG-промпт: отвечай ТОЛЬКО на основе контекста
RAG_SYSTEM = """Ты повар-ассистент. Отвечай кратко, только фактами из предоставленного контекста.
Если в контексте нет ответа — пиши 'Нет данных'.
Не выдумывай ингредиенты и пропорции.
Отвечай на русском."""

def rag_answer(question, k=3):
    context, results = build_context(question, k=k)

    user_prompt = f"""Контекст:
{context}

Вопрос: {question}

Ответь кратко на основе контекста."""

    response = client.chat.completions.create(
        model=llm_client.OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": RAG_SYSTEM},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
    )

    answer = response.choices[0].message.content
    sources = [r["meta"]["recipe_title"] for r in results]
    return answer, sources


# тест
q = "Расскажи про шоколадный торт"
answer, sources = rag_answer(q)
print(f"Q: {q}")
print()
print(f"A: {answer}")
print()
print(f"Sources: {sources}")

Q: Расскажи про шоколадный торт

A: В контексте есть два рецепта шоколадных тортов:

**1. Ultimate chocolate cake**  
- Ингредиенты: 3 oz (≈85 г) несладкого шоколада, 1 ст. л. несоленого масла, 2 ⅓ ст. л. плотно утрамбованного светло-коричневого сахара, 2 ⅓ ст. л. просеянной муки для торта, 2/3 ст. л. пахты, 1 ст. л. кипятка, 1 ч. л. ванили, 3 крупных яйца, 2 ч. л. соды, 1 ч. л. соли.  
- Приготовление: растопить шоколад на водяной бане, охладить; взбить масло с сахаром, добавить яйца по одному, затем остальные сухие ингредиенты, в конце влить пахту и кипяток, добавить ваниль. Выпекать при 350 °F (≈177 °C) в смазанных маслом и присыпанных мукой формах.

**2. Betsy’s chocolate fudge cake**  
- Ингредиенты: 3 sq (≈85 г) несладкого шоколада, 2 ¼ ст. л. просеянной муки, 2 ч. л. разрыхлителя, ½ ч. л. соли, ½ ст. л. масла, 2 ¼ ст. л. плотно утрамбованного светло‑коричневого сахара, 3 яйца, 1 ½ ч. л. ванили, 1 ст. л. сметаны, 1 ст. л. кипятка.  
- Приготовление: растопить шоколад над горячей 

In [14]:
# тот же вопрос, но БЕЗ контекста
def llm_no_rag(question):
    response = client.chat.completions.create(
        model=llm_client.OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": RAG_SYSTEM},
            {"role": "user", "content": question},
        ],
        temperature=0,
    )
    return response.choices[0].message.content


q = "Расскажи про шоколадный торт из наших рецептов"
print(f"Q: {q}")
print()
print(" БЕЗ RAG ")
print(llm_no_rag(q))
print()
print(" С RAG ")
answer, sources = rag_answer(q)
print(answer)
print()
print("Sources:", sources)

Q: Расскажи про шоколадный торт из наших рецептов

 БЕЗ RAG 
Нет данных.

 С RAG 
**Шоколадные торты из представленных рецептов**

1. **Ultimate chocolate cake**  
   - Ингредиенты: 3 oz (≈85 г) несладкого шоколада, 1 ст. ч. несоленого масла, 2 ⅓ ст. ч. светло-коричневого сахара, 2 ⅓ ст. ч. просеянной муки для торта, 2/3 ст. ч. пахты, 1 ст. ч. кипятка, 1 ч. л. ванили, 3 крупных яйца, 2 ч. л. соды, 1 ч. л. соли.  
   - Приготовление: растопить шоколад, охладить; кремом взбить масло с сахаром, добавить яйца, затем остальные сухие и жидкие ингредиенты; выпекать при 350 °F (≈177 °C).

2. **Chocolate mochadot cake**  
   - Ингредиенты: 1 ¾ ст. ч. просеянной муки, ½ ч. л. разрыхлителя, 1 ч. л. соды, 1 ч. л. соли, 1 ч. л. корицы, ½ ст. ч. какао, 1 ½ ст. ч. сахара, ½ ст. ч. шортенинга, ½ ст. ч. пахты, ½ ст. ч. сильного охлаждённого кофе, 2 яйца, 1 ч. л. ванили.  
   - Приготовление: сухие ингредиенты просеять, добавить шортенинг, пахту и кофе, взбить 2 минуты; добавить оставшуюся пахту/сметану